# 35 · Load: NYS DOL unemployment insurance beneficiaries → `staging`

**Source card:** `nysdol_ui.md`. Context series: how many people draw regular unemployment benefits, by county, every month since 2001.

| Table | One row = | Expected rows |
|---|---|---|
| `stg_nysdol__ui_beneficiaries` | year × month × county (62 NY counties + out-of-state + unknown) | 19,712 |

NYS DOL "Unemployment Insurance Beneficiaries and Benefit Amounts Paid: Beginning 2001" (data.ny.gov `xbjp-8sra`), downloaded 2026-10-12. Regular UI only (UI, UCFE, UCX, Shared Work, SEAP); federal extensions and 599.2 training are excluded, and the page doesn't say whether pandemic programs (PUA, PEUC) are in, so the 2020 peak should be read as regular UI. Beneficiaries are rounded to the nearest 100 and dollars to the nearest $10,000 (publisher). No license is stated; attribution to NYS DOL. All 62 counties are kept (the file is small); NYC = the 5 counties in region "New York City".

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from staging import connect, raw_files, stage

pd.set_option("display.max_columns", 40, "display.width", 220)
NB = "35-load-nysdol-ui.ipynb"
conn = connect()

row = raw_files("nysdol_ui_beneficiaries").iloc[-1]
u = pd.read_csv(row.path, dtype=str, keep_default_na=False)
u.columns = [c.strip() for c in u.columns]
print(u.shape, u.columns.tolist())

(19712, 6) ['Year', 'Month', 'Region', 'County', 'Beneficiaries', 'Benefit Amounts (Dollars)']


## 1. Checks: complete grid, numbers, rounding

In [2]:
for c in ["Year", "Month", "Region", "County"]:
    assert (u[c] == u[c].str.strip()).all() and (u[c] != "").all(), c
u["year"] = u.Year.astype(int).astype("Int64")
u["month"] = u.Month.astype(int).astype("Int64")
assert u.month.between(1, 12).all()
u["beneficiaries"] = pd.to_numeric(u.Beneficiaries, errors="raise").astype("Int64")
u["benefit_amount_usd"] = pd.to_numeric(u["Benefit Amounts (Dollars)"], errors="raise").astype("Int64")
assert (u.beneficiaries >= 0).all() and (u.benefit_amount_usd >= 0).all()
print("beneficiaries multiple of 100:", (u.beneficiaries % 100 == 0).mean(), "| dollars multiple of 10,000:", (u.benefit_amount_usd % 10_000 == 0).mean())
assert not u.duplicated(["year", "month", "County"]).any()
assert u.groupby("County").Region.nunique().max() == 1, "a county sits in two regions"
g = u.groupby(["year", "month"]).size()
print(f"months: {len(g)} ({g.index.min()} -> {g.index.max()}) | areas per month: {g.unique().tolist()}")
assert (g == 64).all()
months = pd.period_range(f"{g.index.min()[0]}-{g.index.min()[1]:02d}", f"{g.index.max()[0]}-{g.index.max()[1]:02d}", freq="M")
assert len(months) == len(g), "a month is missing"
nyc = u[u.Region == "New York City"]
assert sorted(nyc.County.unique()) == ["Bronx", "Kings", "New York", "Queens", "Richmond"]

beneficiaries multiple of 100: 1.0 | dollars multiple of 10,000: 1.0
months: 308 ((np.int64(2001), np.int64(1)) -> (np.int64(2026), np.int64(8))) | areas per month: [64]


## 2. A first look: NYC beneficiaries, average per month by year

Monthly counts, so the yearly view is the average month. 2020 is the COVID peak; 2026 runs through August only.

In [3]:
y = nyc.groupby(["year", "month"]).beneficiaries.sum().groupby("year").mean().round(-2)
st = u.groupby(["year", "month"]).beneficiaries.sum().groupby("year").mean().round(-2)
display(pd.DataFrame({"nyc_avg_month": y, "statewide_avg_month": st, "nyc_share": (y / st).round(3)}).loc[2015:])

,nyc_avg_month,statewide_avg_month,nyc_share
year,,,
2015,76500.0,199900.0,0.383
2016,71700.0,185700.0,0.386
2017,70400.0,177900.0,0.396
2018,65300.0,165300.0,0.395
2019,62600.0,157100.0,0.398
2020,452700.0,946300.0,0.478
2021,157900.0,340700.0,0.463
2022,59800.0,143700.0,0.416
2023,72500.0,168500.0,0.43


## 3. Stage

In [4]:
stg = (u.rename(columns={"Region": "region", "County": "county"})
        .assign(is_nyc=lambda d: d.region == "New York City", source_file=row.file)
        [["year", "month", "region", "county", "is_nyc", "beneficiaries", "benefit_amount_usd", "source_file"]]
        .sort_values(["year", "month", "region", "county"]).reset_index(drop=True))
display(stage(stg, "stg_nysdol__ui_beneficiaries", ["year", "month", "county"], expected_rows=19_712, source_key="nysdol_ui_beneficiaries",
              notebook=NB, conn=conn,
              comment="NYS DOL regular UI beneficiaries and benefit dollars paid, by county (62 + out-of-state + unknown), monthly since 2001. Rounded to 100 people / $10,000 by the publisher."))
if conn is not None:
    conn.close()

  stg_nysdol__ui_beneficiaries: 19,712 rows x 8 cols, key ['year', 'month', 'county'] unique | local data/interim/staging/stg_nysdol__ui_beneficiaries.parquet | DB: 19712


,table,rows,cols,key,source_key,notebook,in_db,loaded_utc
0,stg_nysdol__ui_beneficiaries,19712,8,year + month + county,nysdol_ui_beneficiaries,35-load-nysdol-ui.ipynb,19712,2026-10-14T22:11:42+00:00
